# Music Annotation Project Notebook 62
## Page258 59R1-only / 59R2-retained probability-mass audit

**Runtime:** CPU. In Colab, select **Runtime → Run all**.

This read-only notebook uses the compact scored parquet and four Page258 PNGs produced by Notebook 61:

- `observed.png`
- `59r1_shadow.png`
- `59r2_shadow.png`
- `decision_difference.png`

It does not require binary SAFE_REMOVE mask PNGs. It takes the final 59R1 decision from parquet `safe_remove`, reconstructs the final 59R2 decision from parquet `p_remove_59r2` plus the frozen 59R2 coherence contract, and verifies the reconstruction against the saved shadow images before analysing probabilities.

### Primary question
For Page258 pixels that 59R1 removes but 59R2 retains, is 59R2 probability mass concentrated immediately below `0.995`, rejected mainly by coherence after passing `0.995`, or substantially farther below the threshold?

### Guardrails
- no model scoring, fitting, refitting or recalibration;
- no threshold or coherence changes;
- no source parquet or image modification;
- `observed_value` is recovered exactly from `observed.png[y, x]`;
- observed intensity is descriptive evidence, not annotation truth;
- the large source parquet is excluded from the upload ZIP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, platform, shutil, sys, zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
NB61_ROOT = ROOT / '59r2_annotated_only_shadow_test'
NB61_RUN_OVERRIDE = None
R59R2_OVERRIDE = None
PAGE = 'Page258'

OUTPUT_ROOT = ROOT / 'notebook62_page258_probability_audit'
RUN = OUTPUT_ROOT / (datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook62_page258_probability_audit')
TABLES, CHARTS, PAGE_OUT, CROPS, AUDIT, UPLOAD = [RUN/x for x in ['tables','charts','page258','page258/component_crops','audit','upload']]
for p in [RUN,TABLES,CHARTS,PAGE_OUT,CROPS,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

PROBABILITY_BANDS = [
    (0.995,1.0000001,'A_GE_0995','>= 0.995'),
    (0.990,0.995,'B_0990_0995','0.990-0.995'),
    (0.980,0.990,'C_0980_0990','0.980-0.990'),
    (0.950,0.980,'D_0950_0980','0.950-0.980'),
    (0.900,0.950,'E_0900_0950','0.900-0.950'),
    (0.500,0.900,'F_0500_0900','0.500-0.900'),
    (0.000,0.500,'G_LT_0500','< 0.500'),
]
INTENSITY_BANDS=[(-np.inf,80,'very_dark_<80'),(80,140,'dark_80_140'),(140,180,'midtone_140_180'),(180,210,'light_180_210'),(210,235,'very_light_210_235'),(235,np.inf,'paper_like_>=235')]
MAX_COMPONENT_CROPS=24
CROP_MARGIN=55
OVERLAY_ALPHA=.62
print('Output:',RUN)

## 2. Helpers and source discovery

In [ ]:
def require(v,msg):
    if not v: raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def json_number(*values,default=None):
    for v in values:
        if v is not None: return v
    return default

NB61_REQUIRED=[
 'tables/59r2_scored_reviewed_cohort.parquet','experiment_identity.json',
 f'pages/{PAGE}/observed.png',f'pages/{PAGE}/59r1_shadow.png',
 f'pages/{PAGE}/59r2_shadow.png',f'pages/{PAGE}/decision_difference.png']
NB61=Path(NB61_RUN_OVERRIDE) if NB61_RUN_OVERRIDE else latest_complete(NB61_ROOT,NB61_REQUIRED)
require(NB61 is not None,'Complete Notebook 61 run not found. Set NB61_RUN_OVERRIDE.')

PARQUET_PATH=NB61/'tables'/'59r2_scored_reviewed_cohort.parquet'
OBSERVED_PATH=NB61/'pages'/PAGE/'observed.png'
SHADOW1_PATH=NB61/'pages'/PAGE/'59r1_shadow.png'
SHADOW2_PATH=NB61/'pages'/PAGE/'59r2_shadow.png'
DIFF_PATH=NB61/'pages'/PAGE/'decision_difference.png'
IDENTITY_PATH=NB61/'experiment_identity.json'
identity61=json.loads(IDENTITY_PATH.read_text())

if R59R2_OVERRIDE is not None:
    R59R2=Path(R59R2_OVERRIDE)
elif identity61.get('r59r2') and Path(identity61['r59r2']).exists():
    R59R2=Path(identity61['r59r2'])
else:
    R59R2=latest_complete(ROOT,['bundle/policy.json','bundle/runtime_contract.json','bundle/predictor_schema.json'])
require(R59R2 is not None,'Frozen 59R2 run not found. Set R59R2_OVERRIDE.')
print('Notebook 61:',NB61)
print('Frozen 59R2:',R59R2)

## 3. Load the frozen 59R2 coherence contract

In [ ]:
BUNDLE=R59R2/'bundle'
policy=json.loads((BUNDLE/'policy.json').read_text())
contract=json.loads((BUNDLE/'runtime_contract.json').read_text())

THRESHOLD=float(json_number(contract.get('threshold'),contract.get('remove_threshold'),policy.get('threshold'),policy.get('remove_threshold')))
CONNECTIVITY=int(json_number(contract.get('connectivity'),policy.get('connectivity'),default=8))
MINIMUM_RUN=int(json_number(contract.get('minimum_run'),contract.get('min_run'),policy.get('minimum_run'),default=2))
REGION_MEAN_THRESHOLD=float(json_number(contract.get('region_mean_threshold'),policy.get('region_mean_threshold'),default=THRESHOLD))

require(THRESHOLD==0.995,f'Expected frozen 59R2 threshold 0.995, found {THRESHOLD}. Do not silently continue.')
require(CONNECTIVITY in [4,8],f'Invalid connectivity: {CONNECTIVITY}')

contract_record={'r59r2':str(R59R2),'policy_id':policy.get('policy_id'),'compatibility_id':policy.get('compatibility_id'),'threshold':THRESHOLD,'connectivity':CONNECTIVITY,'minimum_run':MINIMUM_RUN,'region_mean_threshold':REGION_MEAN_THRESHOLD,'policy_sha256':sha256_file(BUNDLE/'policy.json'),'runtime_contract_sha256':sha256_file(BUNDLE/'runtime_contract.json')}
atomic_json(AUDIT/'resolved_59r2_coherence_contract.json',contract_record)
print(json.dumps(contract_record,indent=2))

## 4. Load Page258 and recover observed intensity

In [ ]:
try:
    page=pd.read_parquet(PARQUET_PATH,filters=[('page','==',PAGE)])
except Exception as e:
    print('Predicate pushdown unavailable:',repr(e)); page=pd.DataFrame()
if page.empty:
    all_rows=pd.read_parquet(PARQUET_PATH); page=all_rows[all_rows.page.astype(str)==PAGE].copy(); del all_rows

required={'page','y','x','component_key','paper_value','p_remove','safe_remove','raw_remove_59r2','p_remove_59r2'}
missing=required-set(page.columns)
require(not missing,f'Compact Notebook 61 parquet lacks: {sorted(missing)}')
page=page.reset_index(drop=True); page['page']=page.page.astype(str)
for c in ['y','x','component_key']: page[c]=pd.to_numeric(page[c],errors='raise').astype(np.int32)
for c in ['paper_value','p_remove','raw_remove_59r2','p_remove_59r2']: page[c]=pd.to_numeric(page[c],errors='raise').astype(np.float64)
page['safe_remove']=page.safe_remove.astype(bool)
require(not page.duplicated(['y','x']).any(),'Duplicate Page258 coordinates.')
require(np.isfinite(page[['p_remove','raw_remove_59r2','p_remove_59r2']].to_numpy()).all(),'Non-finite score found.')
require(((page.p_remove_59r2>=0)&(page.p_remove_59r2<=1)).all(),'59R2 probability outside [0,1].')

observed=cv2.imread(str(OBSERVED_PATH),cv2.IMREAD_GRAYSCALE)
shadow1=cv2.imread(str(SHADOW1_PATH),cv2.IMREAD_GRAYSCALE)
shadow2=cv2.imread(str(SHADOW2_PATH),cv2.IMREAD_GRAYSCALE)
saved_diff=cv2.imread(str(DIFF_PATH),cv2.IMREAD_GRAYSCALE)
require(all(x is not None for x in [observed,shadow1,shadow2,saved_diff]),'One or more Page258 PNGs are unreadable.')
require(observed.shape==shadow1.shape==shadow2.shape==saved_diff.shape,'Page258 PNG shape mismatch.')
h,w=observed.shape; y=page.y.to_numpy(); x=page.x.to_numpy()
require((y>=0).all() and (y<h).all() and (x>=0).all() and (x<w).all(),'Parquet coordinate outside observed image.')
page['observed_value']=observed[y,x].astype(np.float32)
atomic_json(AUDIT/'observed_value_reconstruction.json',{'page':PAGE,'source':str(OBSERVED_PATH),'source_sha256':sha256_file(OBSERVED_PATH),'method':'exact observed[y,x] lookup','resizing':False,'interpolation':False,'normalisation':False,'truth_source':False,'rows':len(page)})
print('Page258 rows:',f'{len(page):,}','shape:',observed.shape)

## 5. Reconstruct 59R2 page-global coherence and replay saved outputs

In [ ]:
def coherent_decision(y,x,probability,height,width,threshold,connectivity,minimum_run,region_mean_threshold):
    y=np.asarray(y,np.int32); x=np.asarray(x,np.int32); probability=np.asarray(probability,np.float32)
    provisional=probability>=float(threshold)
    candidate=np.zeros((height,width),np.uint8); candidate[y[provisional],x[provisional]]=1
    n,labels,stats,_=cv2.connectedComponentsWithStats(candidate,int(connectivity))
    pmap=np.full((height,width),np.nan,np.float32); pmap[y,x]=probability
    safe=np.zeros((height,width),bool)
    for cid in range(1,n):
        region=labels==cid; size=int(stats[cid,cv2.CC_STAT_AREA]); vals=pmap[region]; finite=vals[np.isfinite(vals)]
        if size>=int(minimum_run) and len(finite)==size and float(finite.mean())>=float(region_mean_threshold) and float(finite.min())>=float(threshold): safe[region]=True
    return safe[y,x],provisional

safe1=page.safe_remove.to_numpy(bool)
safe2,provisional2=coherent_decision(y,x,page.p_remove_59r2.to_numpy(),h,w,THRESHOLD,CONNECTIVITY,MINIMUM_RUN,REGION_MEAN_THRESHOLD)
paper=page.paper_value.to_numpy()

replay1=observed.copy(); replay2=observed.copy()
replay1[y[safe1],x[safe1]]=np.clip(np.rint(paper[safe1]),0,255).astype(np.uint8)
replay2[y[safe2],x[safe2]]=np.clip(np.rint(paper[safe2]),0,255).astype(np.uint8)

shadow1_diff=int(np.sum(replay1!=shadow1)); shadow2_diff=int(np.sum(replay2!=shadow2))
require(shadow1_diff==0,f'59R1 shadow replay failed: {shadow1_diff:,} differing pixels.')
require(shadow2_diff==0,f'59R2 shadow replay failed: {shadow2_diff:,} differing pixels. Check the exact frozen coherence contract/run.')

# Bug-fixed Notebook 61 may define decision_difference as decision XOR. Earlier code variants used rendered-byte XOR.
decision_xor=np.zeros_like(observed); decision_xor[y[safe1!=safe2],x[safe1!=safe2]]=255
rendered_xor=np.where(replay1!=replay2,255,0).astype(np.uint8)
match_decision=bool(np.array_equal(saved_diff,decision_xor)); match_rendered=bool(np.array_equal(saved_diff,rendered_xor))
require(match_decision or match_rendered,'decision_difference.png matches neither decision XOR nor rendered-byte XOR.')
difference_semantics='decision_xor' if match_decision else 'rendered_byte_xor'

replay={'59r1_shadow_exact':True,'59r2_shadow_exact':True,'decision_difference_exact':True,'decision_difference_semantics':difference_semantics,'59r1_safe_remove':int(safe1.sum()),'59r2_safe_remove':int(safe2.sum()),'changed_decisions':int((safe1!=safe2).sum()),'rendered_difference_pixels':int((replay1!=replay2).sum())}
atomic_json(AUDIT/'page258_exact_replay.json',replay)
print(json.dumps(replay,indent=2))

## 6. Define populations and probability/intensity bands

In [ ]:
page['safe_remove_59r1']=safe1; page['safe_remove_59r2']=safe2; page['passes_59r2_pixel_threshold']=provisional2
page['population']=np.select([safe1&~safe2,safe1&safe2,~safe1&safe2],['59R1_ONLY','BOTH','59R2_ONLY'],default='NEITHER')
page['coherence_rejected_after_threshold']=provisional2&~safe2

def pband(v):
    for lo,hi,bid,label in PROBABILITY_BANDS:
        if lo<=v<hi:return bid
    return 'UNASSIGNED'
def iband(v):
    for lo,hi,label in INTENSITY_BANDS:
        if lo<=v<hi:return label
    return 'UNASSIGNED'
page['probability_band']=[pband(v) for v in page.p_remove_59r2]
page['intensity_band']=[iband(v) for v in page.observed_value]
require(not (page.probability_band=='UNASSIGNED').any(),'Unassigned probability band.')

disagreement=page[page.population=='59R1_ONLY'].copy()
require(len(disagreement)>0,'No 59R1-only Page258 pixels.')
page.to_parquet(TABLES/'page258_scored_with_reconstructed_decisions.parquet',index=False)
disagreement.to_parquet(TABLES/'page258_59r1_only_59r2_retained.parquet',index=False)
print(page.population.value_counts().to_string())

## 7. Exact probability mass, quantiles and diagnostic interpretation

In [ ]:
qgrid=[0,.001,.01,.05,.10,.25,.50,.75,.90,.95,.975,.99,.995,.999,1]
pops={'59R1_ONLY':page.population=='59R1_ONLY','BOTH':page.population=='BOTH','59R2_ONLY':page.population=='59R2_ONLY','NEITHER':page.population=='NEITHER','59R1_ONLY_OBS_GE_180':(page.population=='59R1_ONLY')&(page.observed_value>=180),'59R1_ONLY_OBS_GE_210':(page.population=='59R1_ONLY')&(page.observed_value>=210)}
rows=[]
for name,m in pops.items():
    vals=page.loc[m,'p_remove_59r2']
    if len(vals):
        for q,v in vals.quantile(qgrid).items(): rows.append({'population':name,'rows':len(vals),'quantile':q,'p_remove_59r2':v})
quantiles=pd.DataFrame(rows); quantiles.to_csv(TABLES/'probability_quantiles.csv',index=False)

bands=(disagreement.groupby('probability_band',as_index=False).agg(pixels=('p_remove_59r2','size'),mean_probability=('p_remove_59r2','mean'),median_probability=('p_remove_59r2','median'),mean_observed=('observed_value','mean'),components=('component_key','nunique')))
order={b[2]:i for i,b in enumerate(PROBABILITY_BANDS)}; bands['order']=bands.probability_band.map(order); bands=bands.sort_values('order').drop(columns='order'); bands['fraction']=bands.pixels/len(disagreement); bands.to_csv(TABLES/'59r1_only_probability_bands.csv',index=False)

cross=(disagreement.groupby(['intensity_band','probability_band'],as_index=False).agg(pixels=('p_remove_59r2','size'),components=('component_key','nunique'),mean_probability=('p_remove_59r2','mean'))); cross.to_csv(TABLES/'intensity_by_probability_band.csv',index=False)

summary={'page':PAGE,'59r1_only_pixels':len(disagreement),'below_0_500':int((disagreement.p_remove_59r2<.5).sum()),'0_500_to_0_900':int(((disagreement.p_remove_59r2>=.5)&(disagreement.p_remove_59r2<.9)).sum()),'0_900_to_0_950':int(((disagreement.p_remove_59r2>=.9)&(disagreement.p_remove_59r2<.95)).sum()),'0_950_to_0_980':int(((disagreement.p_remove_59r2>=.95)&(disagreement.p_remove_59r2<.98)).sum()),'0_980_to_0_990':int(((disagreement.p_remove_59r2>=.98)&(disagreement.p_remove_59r2<.99)).sum()),'0_990_to_0_995':int(((disagreement.p_remove_59r2>=.99)&(disagreement.p_remove_59r2<.995)).sum()),'at_or_above_0_995_but_coherence_rejected':int((disagreement.p_remove_59r2>=.995).sum()),'observed_ge_180':int((disagreement.observed_value>=180).sum()),'observed_ge_210':int((disagreement.observed_value>=210).sum()),'median':float(disagreement.p_remove_59r2.median()),'p90':float(disagreement.p_remove_59r2.quantile(.9)),'p95':float(disagreement.p_remove_59r2.quantile(.95)),'p99':float(disagreement.p_remove_59r2.quantile(.99)),'maximum':float(disagreement.p_remove_59r2.max())}
near=(summary['0_990_to_0_995']+summary['at_or_above_0_995_but_coherence_rejected'])/len(disagreement)
far=(summary['below_0_500']+summary['0_500_to_0_900'])/len(disagreement)
if near>=.5: interpretation='Most disagreement mass is at or immediately below 0.995; threshold/calibration/coherence is the leading diagnostic issue.'
elif far>=.5: interpretation='Most disagreement mass is below 0.90; representation/ranking is the leading diagnostic issue, not a small threshold adjustment.'
else: interpretation='The disagreement mass is mixed; interpret the spatial bands and component crops rather than attributing the result to one threshold effect.'
summary['diagnostic_interpretation']=interpretation
atomic_json(RUN/'page258_probability_mass_summary.json',summary)
print(json.dumps(summary,indent=2)); display(bands); display(quantiles[quantiles.population.str.startswith('59R1_ONLY')])

## 8. Histogram, upper tail, CDF and band dashboard

In [ ]:
vals=disagreement.p_remove_59r2.to_numpy(); light=disagreement.loc[disagreement.observed_value>=180,'p_remove_59r2'].to_numpy(); vl=disagreement.loc[disagreement.observed_value>=210,'p_remove_59r2'].to_numpy()
fig,ax=plt.subplots(2,2,figsize=(14,10))
ax[0,0].hist(vals,bins=np.linspace(0,1,101)); ax[0,0].axvline(THRESHOLD,color='black',ls='--'); ax[0,0].set(title='All 59R1-only pixels',xlabel='59R2 p_remove',ylabel='Pixels')
fine=np.unique(np.r_[np.linspace(.9,.98,33),np.linspace(.98,.995,31),np.linspace(.995,1.000001,21)]); ax[0,1].hist(vals[vals>=.9],bins=fine,color='#ED7D31'); ax[0,1].axvline(THRESHOLD,color='black',ls='--'); ax[0,1].set(title='Upper tail >= 0.90',xlabel='59R2 p_remove',ylabel='Pixels')
for a,label in [(vals,'all'),(light,'observed >=180'),(vl,'observed >=210')]:
    if len(a): s=np.sort(a); ax[1,0].plot(s,np.arange(1,len(s)+1)/len(s),label=f'{label} ({len(s):,})')
ax[1,0].axvline(THRESHOLD,color='black',ls='--'); ax[1,0].set(title='Empirical CDF',xlabel='59R2 p_remove',ylabel='Fraction <= p'); ax[1,0].legend()
ax[1,1].bar(bands.probability_band,bands.pixels); ax[1,1].set(title='Fine band counts',xlabel='Band',ylabel='Pixels'); ax[1,1].tick_params(axis='x',rotation=45)
for i,v in enumerate(bands.pixels): ax[1,1].text(i,v,f'{v:,}',ha='center',va='bottom',fontsize=8)
plt.tight_layout(); plt.savefig(CHARTS/'probability_mass_dashboard.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)

## 9. Whole-page spatial probability overlays

In [ ]:
BGR={'A_GE_0995':(0,190,0),'B_0990_0995':(0,255,150),'C_0980_0990':(0,255,255),'D_0950_0980':(0,165,255),'E_0900_0950':(255,140,0),'F_0500_0900':(255,0,180),'G_LT_0500':(80,80,80)}
LABEL={b[2]:b[3] for b in PROBABILITY_BANDS}
def blend(gray,masks):
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR); colour=base.copy(); active=np.zeros(gray.shape,bool)
    for m,c in masks: colour[m]=c; active|=m
    out=base.copy(); out[active]=cv2.addWeighted(base,1-OVERLAY_ALPHA,colour,OVERLAY_ALPHA,0)[active]; return out
def legend(img,title,items):
    h0,w0=img.shape[:2]; banner=82; out=np.full((h0+banner,w0,3),255,np.uint8); out[banner:]=img; cv2.putText(out,title,(12,27),cv2.FONT_HERSHEY_SIMPLEX,.62,(25,25,25),1,cv2.LINE_AA); xx=12
    for label,c in items:
        cv2.rectangle(out,(xx,52),(xx+18,70),c,-1); cv2.putText(out,label,(xx+24,68),cv2.FONT_HERSHEY_SIMPLEX,.4,(25,25,25),1,cv2.LINE_AA); xx+=145
    return out

def overlay_for(frame,title):
    masks=[]; items=[]
    for bid in [b[2] for b in PROBABILITY_BANDS]:
        m=np.zeros(observed.shape,bool); q=frame[frame.probability_band==bid]; m[q.y.to_numpy(int),q.x.to_numpy(int)]=True; masks.append((m,BGR[bid])); items.append((LABEL[bid],BGR[bid]))
    return legend(blend(observed,masks),title,items)

all_overlay=overlay_for(disagreement,'Page258: 59R1 removes / 59R2 retains — 59R2 p_remove')
light_overlay=overlay_for(disagreement[disagreement.observed_value>=180],'Page258: observed >=180 subset — descriptive, not truth')
coh=np.zeros(observed.shape,bool); q=disagreement[disagreement.p_remove_59r2>=THRESHOLD]; coh[q.y.to_numpy(int),q.x.to_numpy(int)]=True
coh_overlay=legend(blend(observed,[(coh,(0,0,255))]),'Page258: p_remove >=0.995 but final 59R2 retained',[('coherence rejected',(0,0,255))])
cv2.imwrite(str(PAGE_OUT/'59r1_only_probability_band_overlay.png'),all_overlay); cv2.imwrite(str(PAGE_OUT/'light_pixels_probability_band_overlay.png'),light_overlay); cv2.imwrite(str(PAGE_OUT/'above_threshold_but_coherence_rejected.png'),coh_overlay)
display(cv2.cvtColor(all_overlay,cv2.COLOR_BGR2RGB))

## 10. Component concentration and crops

In [ ]:
comp=(disagreement.groupby('component_key',as_index=False).agg(pixels=('p_remove_59r2','size'),y_min=('y','min'),y_max=('y','max'),x_min=('x','min'),x_max=('x','max'),mean_probability=('p_remove_59r2','mean'),median_probability=('p_remove_59r2','median'),p90_probability=('p_remove_59r2',lambda s:s.quantile(.9)),p99_probability=('p_remove_59r2',lambda s:s.quantile(.99)),maximum_probability=('p_remove_59r2','max'),near_threshold_pixels=('p_remove_59r2',lambda s:int(((s>=.99)&(s<.995)).sum())),above_threshold_rejected=('p_remove_59r2',lambda s:int((s>=.995).sum())),light_pixels=('observed_value',lambda s:int((s>=180).sum())),mean_observed=('observed_value','mean'))).sort_values('pixels',ascending=False).reset_index(drop=True)
comp['near_threshold_fraction']=comp.near_threshold_pixels/comp.pixels; comp['light_fraction']=comp.light_pixels/comp.pixels; comp.to_csv(TABLES/'disagreement_by_component.csv',index=False)
full_overlay=blend(observed,[(np.pad(np.zeros((1,1),bool),((0,h-1),(0,w-1))), (0,0,0))]) # initialise safely
# rebuild once without legend
masks=[]
for bid in [b[2] for b in PROBABILITY_BANDS]:
    m=np.zeros(observed.shape,bool); q=disagreement[disagreement.probability_band==bid]; m[q.y.to_numpy(int),q.x.to_numpy(int)]=True; masks.append((m,BGR[bid]))
full_overlay=blend(observed,masks)
rows=[]
for rank,(_,r) in enumerate(comp.head(MAX_COMPONENT_CROPS).iterrows(),1):
    x0=max(0,int(r.x_min)-CROP_MARGIN); y0=max(0,int(r.y_min)-CROP_MARGIN); x1=min(w,int(r.x_max)+CROP_MARGIN+1); y1=min(h,int(r.y_max)+CROP_MARGIN+1)
    left=cv2.cvtColor(observed[y0:y1,x0:x1],cv2.COLOR_GRAY2BGR); right=full_overlay[y0:y1,x0:x1]; pair=np.hstack([left,np.full((left.shape[0],8,3),230,np.uint8),right]); canvas=np.full((pair.shape[0]+48,pair.shape[1],3),255,np.uint8); canvas[48:]=pair
    title=f'#{rank} component={int(r.component_key)} pixels={int(r.pixels):,} median={r.median_probability:.4f} p99={r.p99_probability:.4f} light={r.light_fraction:.1%}'
    cv2.putText(canvas,title,(8,29),cv2.FONT_HERSHEY_SIMPLEX,.44,(25,25,25),1,cv2.LINE_AA); path=CROPS/f'{rank:02d}_component_{int(r.component_key)}.png'; cv2.imwrite(str(path),canvas); rows.append({'rank':rank,'crop':str(path),**r.to_dict()})
pd.DataFrame(rows).to_csv(TABLES/'component_crop_index.csv',index=False); display(comp.head(20))

## 11. Executive summary and identity

In [ ]:
environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
run_identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'Notebook62-Page258-probability-mass-audit-v2','read_only':True,'source_notebook61_run':str(NB61),'source_parquet':str(PARQUET_PATH),'source_parquet_sha256':sha256_file(PARQUET_PATH),'page':PAGE,'policy_id':policy.get('policy_id'),'compatibility_id':policy.get('compatibility_id'),'coherence_contract':contract_record,'exact_replay':replay,'rows':len(page),'59r1_only_rows':len(disagreement),'observed_value_source':'observed.png[y,x]','observed_intensity_is_not_truth':True,'promotion_authority':False,'environment':environment}
atomic_json(RUN/'experiment_identity.json',run_identity); atomic_json(RUN/'software_environment.json',environment)
lines=['NOTEBOOK 62 — PAGE258 PROBABILITY-MASS AUDIT','',f'Source Notebook 61: {NB61}',f'59R2 policy: {policy.get("policy_id")}',f'Page rows: {len(page):,}',f'59R1 removes / 59R2 retains: {len(disagreement):,}',f'Median p_remove_59r2: {summary["median"]:.6f}',f'P90 / P95 / P99: {summary["p90"]:.6f} / {summary["p95"]:.6f} / {summary["p99"]:.6f}',f'0.990-0.995 pixels: {summary["0_990_to_0_995"]:,}',f'>=0.995 but coherence-rejected: {summary["at_or_above_0_995_but_coherence_rejected"]:,}',f'Below 0.900: {summary["below_0_500"]+summary["0_500_to_0_900"]:,}','','INTERPRETATION',interpretation,'','Exact saved-output replay passed before analysis.','Observed intensity is descriptive only, not handwriting truth.','No model, calibration, threshold, coherence rule, source image or source parquet was changed.']
(RUN/'executive_summary.txt').write_text('
'.join(lines)+'
'); print('
'.join(lines))

## 12. Curated upload package

In [ ]:
curated=[RUN/'executive_summary.txt',RUN/'page258_probability_mass_summary.json',RUN/'experiment_identity.json',AUDIT/'resolved_59r2_coherence_contract.json',AUDIT/'observed_value_reconstruction.json',AUDIT/'page258_exact_replay.json',TABLES/'probability_quantiles.csv',TABLES/'59r1_only_probability_bands.csv',TABLES/'intensity_by_probability_band.csv',TABLES/'disagreement_by_component.csv',TABLES/'component_crop_index.csv',CHARTS/'probability_mass_dashboard.png',PAGE_OUT/'59r1_only_probability_band_overlay.png',PAGE_OUT/'light_pixels_probability_band_overlay.png',PAGE_OUT/'above_threshold_but_coherence_rejected.png']+sorted(CROPS.glob('*.png'))
missing=[str(p) for p in curated if not p.is_file()]; require(not missing,f'Missing curated artefacts: {missing}')
manifest=[]
for src in curated:
    rel=src.relative_to(RUN); dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); require(sha256_file(src)==sha256_file(dst),f'Copy hash mismatch: {rel}'); manifest.append({'relative_path':str(rel),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).sort_values('relative_path').to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 62 Page258 probability-mass audit.
The large source parquet is not included.
Exact 59R1 shadow, 59R2 shadow and decision-difference replay passed before analysis.
Observed intensity is descriptive, not annotation truth.
')
(UPLOAD/'upload_file_list.txt').write_text('
'.join(sorted(str(p.relative_to(UPLOAD)) for p in UPLOAD.rglob('*') if p.is_file()))+'
')
zip_path=RUN/'Notebook62_Page258_Probability_Mass_Audit_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file(): zf.write(p,p.relative_to(UPLOAD))
with zipfile.ZipFile(zip_path) as zf: require(zf.testzip() is None,'ZIP integrity failed.')
print('Upload ZIP:',zip_path); print('SHA-256:',sha256_file(zip_path))